# 3. Markov Decision Processes (MDP)

**Kaynak:** Sutton & Barto, *Reinforcement Learning: An Introduction*, 2nd Edition (2018)
- **Bölüm 3: Finite Markov Decision Processes** (Sayfa 47-72)

## İçindekiler
1. Agent-Environment Interface *(s. 47-51)*
2. Goals and Rewards *(s. 53-54)*
3. Returns and Episodes *(s. 54-57)*
4. Policies and Value Functions *(s. 58-62)*
5. Bellman Equations *(s. 58-65)*
6. Optimal Policies *(s. 62-68)*

---
## 3.1 Markov Property

📖 **Referans:** Sutton & Barto, Sayfa 48-49

> *"A state signal that succeeds in retaining all relevant information is said to be Markov, or to have the Markov property."* (s. 48)

### Markov Property (s. 49)

$$P[S_{t+1} | S_t] = P[S_{t+1} | S_1, S_2, ..., S_t]$$

> *"The future is independent of the past given the present."*

Bu özellik, state'in tüm gerekli bilgiyi içerdiği anlamına gelir.

---
## 3.2 MDP Formülasyonu

📖 **Referans:** Sutton & Barto, Sayfa 48-51, Section 3.1

### Dynamics Function (Equation 3.2, s. 48)

> *"The function p defines the dynamics of the MDP."*

$$p(s', r | s, a) \doteq Pr\{S_t=s', R_t=r | S_{t-1}=s, A_{t-1}=a\}$$

### MDP Tuple

Bir MDP şu tuple ile tanımlanır: $(\mathcal{S}, \mathcal{A}, p, r, \gamma)$

| Bileşen | Açıklama | Referans |
|---------|----------|----------|
| $\mathcal{S}$ | State uzayı | s. 48 |
| $\mathcal{A}$ | Action uzayı | s. 48 |
| $p(s',r\|s,a)$ | Dynamics | Eq. 3.2, s. 48 |
| $r(s,a)$ | Expected reward | Eq. 3.5, s. 49 |
| $\gamma$ | Discount factor | s. 55 |

In [ ]:
# Kod Örneği: Grid World MDP
# Referans: Example 3.5 "Gridworld" (s. 60)

import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple, Dict, List

class GridWorldMDP:
    """
    Grid World MDP (Example 3.5, s. 60)
    
    "Figure 3.2 shows a simple finite MDP... The cells of the grid 
    correspond to the states of the environment."
    """
    
    def __init__(self, rows=4, cols=4, terminal_states=None):
        self.rows = rows
        self.cols = cols
        self.n_states = rows * cols
        self.n_actions = 4  # up, right, down, left
        
        # Terminal states (s. 57: "episodes end in... terminal state")
        self.terminal_states = terminal_states or [0, self.n_states - 1]
        
        # Actions: "stochastically move in the four directions" (s. 60)
        self.action_effects = {
            0: (-1, 0),   # up
            1: (0, 1),    # right
            2: (1, 0),    # down
            3: (0, -1)    # left
        }
        self.action_names = ['↑', '→', '↓', '←']
    
    def state_to_pos(self, state: int) -> Tuple[int, int]:
        return state // self.cols, state % self.cols
    
    def pos_to_state(self, row: int, col: int) -> int:
        return row * self.cols + col
    
    def get_transition_prob(self, state: int, action: int, next_state: int) -> float:
        """
        p(s'|s, a) - State transition probability (derived from Eq. 3.4, s. 49)
        Deterministic transitions in this simple gridworld.
        """
        if state in self.terminal_states:
            return 1.0 if next_state == state else 0.0
        
        row, col = self.state_to_pos(state)
        d_row, d_col = self.action_effects[action]
        
        new_row = max(0, min(self.rows - 1, row + d_row))
        new_col = max(0, min(self.cols - 1, col + d_col))
        expected_next = self.pos_to_state(new_row, new_col)
        
        return 1.0 if next_state == expected_next else 0.0
    
    def get_reward(self, state: int, action: int, next_state: int) -> float:
        """
        r(s, a, s') - Reward function
        "reward is -1 on all transitions until the terminal state" (Example 3.5)
        """
        if state in self.terminal_states:
            return 0
        return -1  # "reward of -1 on all transitions" (s. 60)

mdp = GridWorldMDP(rows=4, cols=4)
print(f"States: {mdp.n_states}, Actions: {mdp.n_actions}")
print(f"Terminal states: {mdp.terminal_states}")

In [ ]:
def visualize_grid(mdp, values=None, policy=None, title="Grid World"):
    """Grid world görselleştirme (Figure 3.2 benzeri, s. 60)"""
    fig, ax = plt.subplots(figsize=(8, 8))
    
    for state in range(mdp.n_states):
        row, col = mdp.state_to_pos(state)
        
        # Color terminal states
        color = 'lightgreen' if state in mdp.terminal_states else 'white'
        
        rect = plt.Rectangle((col, mdp.rows - 1 - row), 1, 1, 
                              facecolor=color, edgecolor='black', linewidth=2)
        ax.add_patch(rect)
        
        # Show value
        if values is not None:
            ax.text(col + 0.5, mdp.rows - row - 0.3, f'{values[state]:.1f}',
                   ha='center', va='center', fontsize=12)
        
        # Show policy
        if policy is not None and state not in mdp.terminal_states:
            ax.text(col + 0.5, mdp.rows - row - 0.7, mdp.action_names[policy[state]],
                   ha='center', va='center', fontsize=16)
        
        # State number
        ax.text(col + 0.1, mdp.rows - row - 0.1, str(state),
               ha='left', va='top', fontsize=8, color='gray')
    
    ax.set_xlim(0, mdp.cols)
    ax.set_ylim(0, mdp.rows)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(title, fontsize=14)
    plt.show()

visualize_grid(mdp, title="4x4 Grid World MDP (Example 3.5, s. 60)")

---
## 3.3 Policies and Value Functions

📖 **Referans:** Sutton & Barto, Sayfa 58-62, Section 3.5

### Policy (s. 58)

> *"A policy is a mapping from states to probabilities of selecting each possible action."*

$$\pi(a|s) = P[A_t = a | S_t = s]$$

### State-Value Function (Equation 3.12, s. 58)

> *"The value function of a state s under a policy π, denoted $v_\pi(s)$, is the expected return when starting in s and following π thereafter."*

$$v_\pi(s) \doteq E_\pi[G_t | S_t = s] = E_\pi\left[\sum_{k=0}^{\infty} \gamma^k R_{t+k+1} | S_t = s\right]$$

### Action-Value Function (Equation 3.13, s. 58)

> *"The value of taking action a in state s under a policy π."*

$$q_\pi(s, a) \doteq E_\pi[G_t | S_t = s, A_t = a]$$

### İlişki (s. 59)

$$v_\pi(s) = \sum_a \pi(a|s) q_\pi(s, a)$$

---
## 3.4 Bellman Equations

📖 **Referans:** Sutton & Barto, Sayfa 59-62

> *"A fundamental property of value functions used throughout reinforcement learning and dynamic programming is that they satisfy recursive relationships."* (s. 59)

### Bellman Equation for $v_\pi$ (Equation 3.14, s. 59)

$$v_\pi(s) = \sum_a \pi(a|s) \sum_{s', r} p(s', r|s, a)[r + \gamma v_\pi(s')]$$

> *"It expresses a relationship between the value of a state and the values of its successor states."* (s. 59)

### Bellman Equation for $q_\pi$ (s. 60)

$$q_\pi(s,a) = \sum_{s', r} p(s',r|s,a)\left[r + \gamma \sum_{a'} \pi(a'|s') q_\pi(s', a')\right]$$

In [ ]:
# Kod Örneği: Bellman Equation ile Policy Evaluation
# Referans: Equation 3.14 (s. 59), Example 3.5 (s. 60)

def evaluate_random_policy(mdp, gamma=1.0, theta=1e-6):
    """
    Evaluate uniform random policy using Bellman equation.
    
    From Example 3.5 (s. 60): "under the equiprobable random policy
    (all actions equally likely)"
    
    Uses Equation 3.14 (s. 59) iteratively.
    """
    V = np.zeros(mdp.n_states)
    action_prob = 1.0 / mdp.n_actions  # Uniform random (s. 60)
    
    iteration = 0
    while True:
        delta = 0
        
        for s in range(mdp.n_states):
            if s in mdp.terminal_states:
                continue
            
            v = V[s]
            new_v = 0
            
            # Equation 3.14: Σ_a π(a|s) Σ_{s',r} p(s',r|s,a)[r + γv(s')]
            for a in range(mdp.n_actions):
                for s_next in range(mdp.n_states):
                    p = mdp.get_transition_prob(s, a, s_next)
                    r = mdp.get_reward(s, a, s_next)
                    new_v += action_prob * p * (r + gamma * V[s_next])
            
            V[s] = new_v
            delta = max(delta, abs(v - V[s]))
        
        iteration += 1
        
        if delta < theta:
            break
    
    print(f"Converged in {iteration} iterations (Bellman Eq. 3.14)")
    return V

# Figure 3.2 left (s. 60) - v_π for random policy
V_random = evaluate_random_policy(mdp)
visualize_grid(mdp, values=V_random, 
               title="v_π for Random Policy (Figure 3.2 left, s. 60)")

---
## 3.5 Optimal Value Functions

📖 **Referans:** Sutton & Barto, Sayfa 62-68, Section 3.6

> *"Solving a reinforcement learning task means, roughly, finding a policy that achieves a lot of reward over the long run."* (s. 62)

### Optimal State-Value Function (Equation 3.15, s. 63)

$$v_*(s) \doteq \max_\pi v_\pi(s)$$

> *"the largest expected return achievable by any policy"* (s. 63)

### Optimal Action-Value Function (Equation 3.16, s. 63)

$$q_*(s, a) \doteq \max_\pi q_\pi(s, a)$$

### İlişki (Equation 3.17, s. 63)

$$q_*(s, a) = E[R_{t+1} + \gamma v_*(S_{t+1}) | S_t=s, A_t=a]$$

---
## 3.6 Bellman Optimality Equations

📖 **Referans:** Sutton & Barto, Sayfa 63-66

### Bellman Optimality for $v_*$ (Equation 3.18, s. 63)

$$v_*(s) = \max_a \sum_{s', r} p(s', r|s, a)[r + \gamma v_*(s')]$$

> *"The Bellman optimality equation expresses the fact that the value of a state under an optimal policy must equal the expected return for the best action from that state."* (s. 63)

### Bellman Optimality for $q_*$ (Equation 3.20, s. 64)

$$q_*(s, a) = \sum_{s', r} p(s', r|s, a)[r + \gamma \max_{a'} q_*(s', a')]$$

In [ ]:
# Kod Örneği: Bellman Optimality Equation ile Value Iteration
# Referans: Equation 3.18 (s. 63)

def value_iteration(mdp, gamma=1.0, theta=1e-6):
    """
    Find optimal value function using Bellman Optimality Equation.
    
    Equation 3.18 (s. 63):
    v*(s) = max_a Σ_{s',r} p(s',r|s,a)[r + γv*(s')]
    """
    V = np.zeros(mdp.n_states)
    
    iteration = 0
    while True:
        delta = 0
        
        for s in range(mdp.n_states):
            if s in mdp.terminal_states:
                continue
            
            v = V[s]
            
            # max_a (Bellman Optimality)
            action_values = []
            for a in range(mdp.n_actions):
                q = 0
                for s_next in range(mdp.n_states):
                    p = mdp.get_transition_prob(s, a, s_next)
                    r = mdp.get_reward(s, a, s_next)
                    q += p * (r + gamma * V[s_next])
                action_values.append(q)
            
            V[s] = max(action_values)  # max_a from Eq. 3.18
            delta = max(delta, abs(v - V[s]))
        
        iteration += 1
        
        if delta < theta:
            break
    
    # Extract optimal policy: π*(s) = argmax_a q*(s,a)
    policy = np.zeros(mdp.n_states, dtype=int)
    for s in range(mdp.n_states):
        if s in mdp.terminal_states:
            continue
        
        action_values = []
        for a in range(mdp.n_actions):
            q = 0
            for s_next in range(mdp.n_states):
                p = mdp.get_transition_prob(s, a, s_next)
                r = mdp.get_reward(s, a, s_next)
                q += p * (r + gamma * V[s_next])
            action_values.append(q)
        
        policy[s] = np.argmax(action_values)
    
    print(f"Value Iteration converged in {iteration} iterations")
    return V, policy

V_star, pi_star = value_iteration(mdp)
visualize_grid(mdp, values=V_star, policy=pi_star, 
               title="v* and π* (Bellman Optimality, Eq. 3.18)")

In [ ]:
# Karşılaştırma: Random Policy vs Optimal Policy
# Referans: Figure 3.2 (s. 60)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Random policy (Figure 3.2 left)
ax = axes[0]
for state in range(mdp.n_states):
    row, col = mdp.state_to_pos(state)
    color = 'lightgreen' if state in mdp.terminal_states else 'white'
    rect = plt.Rectangle((col, mdp.rows - 1 - row), 1, 1, 
                          facecolor=color, edgecolor='black', linewidth=2)
    ax.add_patch(rect)
    ax.text(col + 0.5, mdp.rows - row - 0.5, f'{V_random[state]:.1f}',
           ha='center', va='center', fontsize=14)
ax.set_xlim(0, mdp.cols)
ax.set_ylim(0, mdp.rows)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('v_π (Random Policy)\nFigure 3.2 left, s. 60', fontsize=12)

# Optimal policy (Figure 3.2 right)
ax = axes[1]
for state in range(mdp.n_states):
    row, col = mdp.state_to_pos(state)
    color = 'lightgreen' if state in mdp.terminal_states else 'lightblue'
    rect = plt.Rectangle((col, mdp.rows - 1 - row), 1, 1, 
                          facecolor=color, edgecolor='black', linewidth=2)
    ax.add_patch(rect)
    ax.text(col + 0.5, mdp.rows - row - 0.3, f'{V_star[state]:.1f}',
           ha='center', va='center', fontsize=12)
    if state not in mdp.terminal_states:
        ax.text(col + 0.5, mdp.rows - row - 0.7, mdp.action_names[pi_star[state]],
               ha='center', va='center', fontsize=16)
ax.set_xlim(0, mdp.cols)
ax.set_ylim(0, mdp.rows)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('v* and π* (Optimal)\nFigure 3.2 right, s. 60', fontsize=12)

plt.tight_layout()
plt.show()

---
## Özet

Bu notebook'ta öğrendiklerimiz (Sutton & Barto Chapter 3):

| Kavram | Sayfa | Denklem | Açıklama |
|--------|-------|---------|----------|
| Dynamics | s. 48 | Eq. 3.2 | $p(s',r\|s,a)$ |
| State-Value | s. 58 | Eq. 3.12 | $v_\pi(s) = E_\pi[G_t\|S_t=s]$ |
| Action-Value | s. 58 | Eq. 3.13 | $q_\pi(s,a) = E_\pi[G_t\|S_t=s,A_t=a]$ |
| Bellman Eq. | s. 59 | Eq. 3.14 | Recursive value relationship |
| Optimal $v_*$ | s. 63 | Eq. 3.15 | $\max_\pi v_\pi(s)$ |
| Bellman Opt. | s. 63 | Eq. 3.18 | $v_*(s) = \max_a ...$ |

### Anahtar Denklemler

**Bellman Expectation** (Eq. 3.14, s. 59):
$$v_\pi(s) = \sum_a \pi(a|s) \sum_{s',r} p(s',r|s,a)[r + \gamma v_\pi(s')]$$

**Bellman Optimality** (Eq. 3.18, s. 63):
$$v_*(s) = \max_a \sum_{s',r} p(s',r|s,a)[r + \gamma v_*(s')]$$

---
### Sonraki Notebook
**04 - Dynamic Programming** *(Chapter 4, s. 73-92)*: Policy/Value Iteration